## Latent Topic Extraction from Worldwide Eurepoc Cyber Security Incidents
Aim: Determine latent types of cyber operations that emerge from incident descriptions using NLP. 

Data source: "https://zenodo.org/records/14965395/files/eurepoc_dyadic_dataset_0_1.csv?download=1" 

Steps: 
1. Download data
2. Clean incident descriptions
3. Generate sentence embeddings (Sentence-BERT)
4. Generate topics (BERTopic)
5. Distill topics into more readable topic names and groups
6. Save model and topic modelling output

Model output used in cyber_incident_analysis.ipynb to link latent cyber operation types to the intensity of societal impact. 

In [1]:
import pandas as pd
import html
from sentence_transformers import SentenceTransformer
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired
from sklearn.feature_extraction.text import CountVectorizer
from umap import UMAP
import random
import numpy as np
import spacy
from src.nlp_functions import(remove_geo_entities)

# Reproducibility 
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

/Users/heather.iriye/Library/CloudStorage/OneDrive-KarolinskaInstitutet/Documents/GitHub/eurepoc_cyber_security-/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Read in data 
url = "https://zenodo.org/records/14965395/files/eurepoc_dyadic_dataset_0_1.csv?download=1"

dyadic_data_full = pd.read_csv(url)

print(dyadic_data_full.shape)
print(dyadic_data_full.head())

(4296, 57)
   dyad_id initiator_country initiator_alpha_2 receiver_country  \
0        0       Afghanistan                AF      Afghanistan   
1        1       Afghanistan                AF         Pakistan   
2        1       Afghanistan                AF         Pakistan   
3        2           Algeria                DZ          Algeria   
4        2           Algeria                DZ          Algeria   

  receiver_country_alpha_2_code  incident_id  \
0                            AF         4278   
1                            PK          488   
2                            PK          499   
3                            DZ          525   
4                            DZ         1183   

                                                name  \
0  TalibLeaks Hacked and Leaked Documents from Ta...   
1               Afghan Cyber Army attack on Pakistan   
2       Afghan Cyber Army attack on Pakistan Part II   
3            Over-X vs. Algerian ministry of housing   
4               N

In [3]:
# Isolate columns of interest
drop_vars = ["added_to_db","updated_at","attribution_id",\
    "initiator_alpha_2","receiver_country_alpha_2_code",
    "Data theft","Data theft & Doxing","Disruption",
    "Hijacking with Misuse","Hijacking without Misuse",
    "Ransomware","Not available"]

attack_labels = ["Data theft","Data theft & Doxing", \
    "Disruption","Hijacking with Misuse","Hijacking without Misuse",
    "Ransomware","Not available"]
attack_data = dyadic_data_full[["incident_id"] + attack_labels].copy()

dyadic_data = dyadic_data_full.drop(columns=drop_vars,\
    errors="ignore")

print(f"Original shape: {dyadic_data_full.shape}")
print(f"New shape: {dyadic_data.shape}")

print("\nRemaining columns:")
print(dyadic_data.columns.tolist())


Original shape: (4296, 57)
New shape: (4296, 45)

Remaining columns:
['dyad_id', 'initiator_country', 'receiver_country', 'incident_id', 'name', 'description', 'start_date', 'end_date', 'source_disclosure', 'operation_type', 'impact_indicator_score', 'impact_indicator_label', 'unweighted_intensity', 'weighted_intensity', 'number_attributions', 'number_political_responses', 'number_legal_responses', 'casualties', 'initiator_name', 'initiator_category', 'initiator_subcategory', 'receiver_id', 'receiver_name', 'receiver_category', 'receiver_subcategory', 'receiver_regions', 'offline_conflict_issue', 'offline_conflict_name', 'offline_conflict_intensity', 'offline_conflict_intensity_subcode', 'cyber_conflict_issue', 'physical_effects_spatial', 'physical_effects_temporal', 'target_multiplier', 'functional_impact', 'intelligence_impact', 'economic_impact', 'economic_impact_value', 'economic_impact_currency', 'affected_entities', 'affected_entities_value', 'affected_eu_countries', 'affected_eu

In [4]:
# Check for missing values in descriptions
print(dyadic_data["description"].isnull().sum())
print(dyadic_data[dyadic_data["description"]==""])

0
Empty DataFrame
Columns: [dyad_id, initiator_country, receiver_country, incident_id, name, description, start_date, end_date, source_disclosure, operation_type, impact_indicator_score, impact_indicator_label, unweighted_intensity, weighted_intensity, number_attributions, number_political_responses, number_legal_responses, casualties, initiator_name, initiator_category, initiator_subcategory, receiver_id, receiver_name, receiver_category, receiver_subcategory, receiver_regions, offline_conflict_issue, offline_conflict_name, offline_conflict_intensity, offline_conflict_intensity_subcode, cyber_conflict_issue, physical_effects_spatial, physical_effects_temporal, target_multiplier, functional_impact, intelligence_impact, economic_impact, economic_impact_value, economic_impact_currency, affected_entities, affected_entities_value, affected_eu_countries, affected_eu_countries_value, affected_third_countries, affected_third_countries_value]
Index: []

[0 rows x 45 columns]


In [5]:
# Incident-level dataset for NLP
# Descriptions can be duplicated because of the dyadic nature of the dataset. 
# One incident may span several rows if it affected several countries. 
incident_text_data = (
    dyadic_data
    .drop_duplicates(subset=["incident_id"])
    .copy()
)
print(incident_text_data.head())


   dyad_id initiator_country receiver_country  incident_id  \
0        0       Afghanistan      Afghanistan         4278   
1        1       Afghanistan         Pakistan          488   
2        1       Afghanistan         Pakistan          499   
3        2           Algeria          Algeria          525   
4        2           Algeria          Algeria         1183   

                                                name  \
0  TalibLeaks Hacked and Leaked Documents from Ta...   
1               Afghan Cyber Army attack on Pakistan   
2       Afghan Cyber Army attack on Pakistan Part II   
3            Over-X vs. Algerian ministry of housing   
4               North African Fox Espionage campaign   

                                         description           start_date  \
0  On 7 February 2025, a group of hackers, callin...  2024-01-01 00:00:00   
1  Afghan hackers deface six Pakistani government...  2013-07-11 00:00:00   
2  Afghan hackers hack the webpage of the Pakista...  2013-

In [6]:
# Clean description
incident_text_data["description"] = (
    incident_text_data["description"] 
    .fillna("")
    .astype(str)
    .apply(html.unescape)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

In [7]:
# Identify missing descriptions
print(incident_text_data["description"].isna().sum())

# Empty descriptions
print((incident_text_data["description"] == "").sum())

# Length distribution
incident_text_data["n_words"] = (
    incident_text_data["description"]
    .str.split()
    .str.len()
)

print(incident_text_data["n_words"].describe())

incident_text_data.loc[
    incident_text_data["n_words"] < 5,
    ["incident_id", "description"]
].head(20)


0
0
count    2957.000000
mean       80.144065
std        69.850175
min         3.000000
25%        29.000000
50%        65.000000
75%       112.000000
max       975.000000
Name: n_words, dtype: float64


,incident_id,description
1366,361,Taliban website hacked
2433,370,Israeli Government Site Hacked
4058,358,The Unknowns' hack NASA


In [8]:
# Check how many short descriptions would be lost if filtered out
print((incident_text_data["n_words"] < 5).sum())
print((incident_text_data["n_words"] < 10).sum())

3
105


In [9]:
# Filter out descriptions with less than five words
incident_text_data = incident_text_data[incident_text_data["n_words"] > 5]

In [10]:
# Remove geographic terms from text
nlp = spacy.load("en_core_web_sm")

incident_text_data["description_no_geo"] = (
    incident_text_data["description"]
    .apply(lambda x: remove_geo_entities(x, nlp))
)

In [11]:
# Sentence embeddings with BERT
model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

embeddings = model.encode(
    incident_text_data["description_no_geo"].tolist(),
    show_progress_bar=True
)

Batches: 100%|██████████| 92/92 [00:02<00:00, 31.24it/s]


In [12]:
# Generic stopwords 
cyber_stopwords = [
    "attack",
    "attacks",
    "attacker",
    "hack",
    "hacks",
    "hacked",
    "hacker",
    "hackers",
    "hacking",
    "campaign",
    "group",
    "security",
    "incident",
    "reported",
    "report",
    "unknown"
]

vectorizer_model = CountVectorizer(
    stop_words=cyber_stopwords,
    min_df=5,
    ngram_range=(1, 3)
)

In [13]:
# Get rid of stopwords, BERT wasn't using them effectively
#vectorizer_model = CountVectorizer(
#    stop_words="english",
#    min_df=5,
#    ngram_range=(1, 3)
#)

# BERTopic analysis 
# For reproducibility 
umap_model = UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=42
) 

# For more readable labels
representation_model = KeyBERTInspired()


# Build topic model 
topic_model = BERTopic(
    embedding_model=model,
    umap_model=umap_model,
    vectorizer_model=vectorizer_model,
    representation_model=representation_model,
    min_topic_size=20,
    calculate_probabilities=True
)


topics, probs = topic_model.fit_transform(
    incident_text_data["description_no_geo"],
    embeddings
)

topic_model = topic_model.reduce_topics(
    incident_text_data["description_no_geo"],
    nr_topics=10
)

topics = topic_model.topics_

incident_text_data["topic"] = topics
incident_text_data["topic_prob"] = probs.max(axis=1)

In [14]:
# Examine topics
topic_info = topic_model.get_topic_info()
#print(topic_info.head())

topic_info[["Topic", "Count", "Representation"]]\
    .style.set_table_attributes(
        'style="display:block;max-height:500px;overflow:auto;"'
    )

,Topic,Count,Representation
0,-1,1333,"['ransomware', 'compromised', 'threat', 'cyber', 'stolen', 'operation', 'gained access to', 'government', 'gained access', 'data']"
1,0,420,"['the government', 'attacked', 'gov', 'website of the', 'sites', 'cyber', 'website', 'government', 'site', 'anonymous']"
2,1,286,"['malware', 'the malware', 'cyber espionage', 'vulnerabilities', 'malicious', 'compromised', 'threat actors', 'vulnerability', 'espionage', 'threat']"
3,2,211,"['the ransomware', 'ransomware', 'ransom', 'compromised', 'encrypted', 'breach', 'stolen', 'investigation', 'access', 'services']"
4,3,191,"['ddos', 'with ddos', 'hacktivist', 'cyber', 'targeted', 'disrupted', 'disruption', 'disrupted the', 'authorities', 'telegram']"
5,4,189,"['data breach', 'stolen data', 'personal data', 'breached', 'personal information', 'breach', 'compromised', 'the breach', 'data', 'the data']"
6,5,186,"['ransomware', 'cyberattack', 'compromised', 'intrusion', 'the university', 'university of', 'university', 'gained access to', 'investigation', 'gained access']"
7,6,70,"['exploited vulnerability', 'the theft of', 'exploited', 'theft', 'theft of', 'the theft', 'exploit', 'vulnerability in', 'vulnerability', 'stole']"
8,7,28,"['espionage', 'the investigation', 'investigation', 'suspected', 'forensic', 'human rights', 'device', 'devices', 'the phone', 'phones']"
9,8,28,"['malware', 'the malware', 'the nsa', 'espionage', 'nsa', 'cyber', 'backdoor', 'attackers', 'secure', 'infected']"


In [20]:
# Create interpretable topic labels
topic_info = topic_model.get_topic_info()

topic_labels = (
    topic_info[["Topic", "Representation"]]
    .copy()
)

topic_labels["topic_label"] = (
    topic_labels["Representation"]
    .apply(lambda x: ", ".join(x[:5]))
)

# Rebuild dyadic dataset
dyadic_data = (
    dyadic_data_full
    .drop(columns=drop_vars, errors="ignore")
    .copy()
)

# Merge topic assignments
dyadic_data = dyadic_data.merge(
    incident_text_data,
    on="incident_id",
    how="left"
)

# Merge topic labels
dyadic_data = dyadic_data.merge(
    topic_labels[["Topic", "topic_label"]],
    left_on="topic",
    right_on="Topic",
    how="left"
)

# Keep incidents that received a topic
dyadic_data = dyadic_data.dropna(
    subset=["topic"]
).copy()

print("Missing topics:")
print(dyadic_data["topic"].isna().sum())

print("\nTopic counts:")
print(dyadic_data["topic"].value_counts().sort_index())

Missing topics:
0

Topic counts:
topic
-1.0    2035
 0.0     478
 1.0     748
 2.0     233
 3.0     202
 4.0     194
 5.0     189
 6.0      75
 7.0      38
 8.0      89
Name: count, dtype: int64


In [21]:
# drop duplicates
print(
    dyadic_data[
        ["topic", "topic_label"]
    ].drop_duplicates()
    .sort_values("topic")
)

# remove outliers
dyadic_data = dyadic_data[
    dyadic_data["topic"] != -1
].copy()

      topic                                        topic_label
0      -1.0     ransomware, compromised, threat, cyber, stolen
1       0.0  the government, attacked, gov, website of the,...
112     1.0  malware, the malware, cyber espionage, vulnera...
11      2.0  the ransomware, ransomware, ransom, compromise...
97      3.0       ddos, with ddos, hacktivist, cyber, targeted
426     4.0  data breach, stolen data, personal data, breac...
1087    5.0  ransomware, cyberattack, compromised, intrusio...
1119    6.0  exploited vulnerability, the theft of, exploit...
41      7.0  espionage, the investigation, investigation, s...
71      8.0      malware, the malware, the nsa, espionage, nsa


In [22]:
# Create more readable topic names from topic labels
topic_names = {
    -1: "General Cyber Operations",
     0: "Website Defacement & Hacktivism",
     1: "Malware & Cyber Espionage",
     2: "Ransomware & Cyber Extortion",
     3: "DDoS & Service Disruption Campaigns",
     4: "Data Breaches & Information Exposure",
     5: "Institutional Ransomware Attacks",
     6: "Vulnerability Exploitation & Theft",
     7: "Surveillance & Mobile Espionage",
     8: "Government Intelligence Operations"
} 

dyadic_data["topic_name"] = (
    dyadic_data["topic"]
    .map(topic_names)
)

topic_groups = {
    # Hacktivism
    "Website Defacement & Hacktivism":
        "Hacktivism",

    # Disruption
    "DDoS & Service Disruption Campaigns":
        "Disruption Operations",

    # Intrusion
    "Malware & Cyber Espionage":
        "Intrusion Operations",

    "Vulnerability Exploitation & Theft":
        "Intrusion Operations",

    # Information
    "Surveillance & Mobile Espionage":
        "Information Operations",

    "Government Intelligence Operations":
        "Information Operations",

    # Data exposure
    "Data Breaches & Information Exposure":
        "Data Exposure",

    # Financial
    "Ransomware & Cyber Extortion":
        "Financially Motivated Operations",

    "Institutional Ransomware Attacks":
        "Financially Motivated Operations",

    # Catch-all
    "General Cyber Operations":
        "Other"
}

dyadic_data["topic_group"] = (
    dyadic_data["topic_name"]
    .map(topic_groups)
)

print(
    dyadic_data[
        ["topic", "topic_name", "topic_group"]
    ]
    .drop_duplicates()
    .sort_values("topic")
)

print("\nMissing topic names:")
print(dyadic_data["topic_name"].isna().sum())

print("\nMissing topic groups:")
print(dyadic_data["topic_group"].isna().sum())

      topic                            topic_name  \
1       0.0       Website Defacement & Hacktivism   
112     1.0             Malware & Cyber Espionage   
11      2.0          Ransomware & Cyber Extortion   
97      3.0   DDoS & Service Disruption Campaigns   
426     4.0  Data Breaches & Information Exposure   
1087    5.0      Institutional Ransomware Attacks   
1119    6.0    Vulnerability Exploitation & Theft   
41      7.0       Surveillance & Mobile Espionage   
71      8.0    Government Intelligence Operations   

                           topic_group  
1                           Hacktivism  
112               Intrusion Operations  
11    Financially Motivated Operations  
97               Disruption Operations  
426                      Data Exposure  
1087  Financially Motivated Operations  
1119              Intrusion Operations  
41              Information Operations  
71              Information Operations  

Missing topic names:
0

Missing topic groups:
0


In [38]:
print(dyadic_data.info())
#print(dyadic_data["functional_impact_x"].value_counts())

<class 'pandas.DataFrame'>
Index: 2246 entries, 1 to 4287
Data columns (total 97 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   dyad_id_x                             2246 non-null   int64  
 1   initiator_country_x                   2246 non-null   str    
 2   receiver_country_x                    2246 non-null   str    
 3   incident_id                           2246 non-null   int64  
 4   name_x                                2246 non-null   str    
 5   description_x                         2246 non-null   str    
 6   start_date_x                          2246 non-null   str    
 7   end_date_x                            2246 non-null   str    
 8   source_disclosure_x                   2246 non-null   str    
 9   operation_type_x                      2246 non-null   str    
 10  impact_indicator_score_x              2246 non-null   int64  
 11  impact_indicator_label_x         

In [40]:
# Save model 
topic_model.save("data/topic_model/eurepoc_final_model")

incident_text_data[
    ["incident_id", "topic", "topic_prob"]
].to_csv(
    "data/topic_model/eurepoc_topic_assignments.csv",
    index=False
) 

topic_model.get_topic_info().to_csv(
    "data/topic_model/topic_info.csv",
    index=False
)

# Save dyadic_data 
dyadic_data.to_csv("data/eurepoc_dataset/dyadic_data_topics.csv", index=False)

2026-08-20 11:43:36,548 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.
